## Employee Attrition Prediction

# Problem Statement

XYZ Company is experiencing an annual employee attrition rate of approximately 15%, resulting in increased recruitment costs, project delays, and additional onboarding and training efforts. To address this challenge, the objective of this project is to analyze employee-related data and identify the key factors influencing attrition. The insights obtained will support HR and management in implementing targeted retention strategies, while the predictive model will help identify employees who are at a higher risk of leaving the organization.

### Data ingestion

\Importing necessary libraries

In [54]:
import numpy as np
import pandas as pd
import os

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

\dataset description

In [55]:
data_dict = pd.read_excel(r"C:\NG\Employee Attrition Prediction\data\raw\data_dictionary.xlsx")
data_dict

,Variable,Meaning,Levels
0,Age,Age of the employee,NaN
1,Attrition,Whether the employee left in the previous year...,NaN
2,BusinessTravel,How frequently the employees travelled for bus...,NaN
3,Department,Department in company,NaN
4,DistanceFromHome,Distance from home in kms,NaN
5,Education,Education Level,1 'Below College'
6,NaN,NaN,2 'College'
7,NaN,NaN,3 'Bachelor'
8,NaN,NaN,4 'Master'
9,NaN,NaN,5 'Doctor'


# dataset details

-- general_data: core HR record — demographics, job/org attributes, compensation, tenure, and the target Attrition.

-- employee_survey_data: self-reported satisfaction scores (environment, job, work-life balance).

-- manager_survey_data: manager-rated scores (involvement, performance).

-- in_time/out_time: raw daily attendance logs for 2015, source for engineered attendance/overtime features.


\Loading the dataset

In [56]:
general_data = pd.read_excel(r"data/raw/general_data.xlsx")
employee_survey = pd.read_excel(r"data/raw/employee_survey_data.xlsx")
manager_survey = pd.read_excel(r"data/raw/manager_survey_data.xlsx")
in_time = pd.read_excel(r"data/raw/in_time.xlsx")
out_time = pd.read_excel(r"data/raw/out_time.xlsx")

\data inspection

In [57]:
print("Files loaded successfully! \n")
print(f"general      : {general_data.shape[0]} rows, {general_data.shape[1]} columns")
print(f"emp_survey   : {employee_survey.shape[0]} rows, {employee_survey.shape[1]} columns")
print(f"mgr_survey   : {manager_survey.shape[0]} rows, {manager_survey.shape[1]} columns")
print(f"in_time      : {in_time.shape[0]} rows, {in_time.shape[1]} columns")
print(f"out_time     : {out_time.shape[0]} rows, {out_time.shape[1]} columns")

Files loaded successfully! 

general      : 4410 rows, 24 columns
emp_survey   : 4410 rows, 4 columns
mgr_survey   : 4410 rows, 3 columns
in_time      : 4410 rows, 262 columns
out_time     : 4410 rows, 262 columns


\employee general data

In [58]:
general_data.head()

,Age,Attrition,BusinessTravel,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeID,Gender,JobLevel,JobRole,MaritalStatus,MonthlyIncome,NumCompaniesWorked,Over18,PercentSalaryHike,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,YearsAtCompany,YearsSinceLastPromotion,YearsWithCurrManager
0,51,No,Travel_Rarely,Sales,6,2,Life Sciences,1,1,Female,1,Healthcare Representative,Married,131160,1.0,Y,11,8,0,1.0,6,1,0,0
1,31,Yes,Travel_Frequently,Research & Development,10,1,Life Sciences,1,2,Female,1,Research Scientist,Single,41890,0.0,Y,23,8,1,6.0,3,5,1,4
2,32,No,Travel_Frequently,Research & Development,17,4,Other,1,3,Male,4,Sales Executive,Married,193280,1.0,Y,15,8,3,5.0,2,5,0,3
3,38,No,Non-Travel,Research & Development,2,5,Life Sciences,1,4,Male,3,Human Resources,Married,83210,3.0,Y,11,8,3,13.0,5,8,7,5
4,32,No,Travel_Rarely,Research & Development,10,1,Medical,1,5,Male,1,Sales Executive,Single,23420,4.0,Y,12,8,2,9.0,2,6,0,4


In [59]:
general_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 4410 entries, 0 to 4409
Data columns (total 24 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Age                      4410 non-null   int64  
 1   Attrition                4410 non-null   str    
 2   BusinessTravel           4410 non-null   str    
 3   Department               4410 non-null   str    
 4   DistanceFromHome         4410 non-null   int64  
 5   Education                4410 non-null   int64  
 6   EducationField           4410 non-null   str    
 7   EmployeeCount            4410 non-null   int64  
 8   EmployeeID               4410 non-null   int64  
 9   Gender                   4410 non-null   str    
 10  JobLevel                 4410 non-null   int64  
 11  JobRole                  4410 non-null   str    
 12  MaritalStatus            4410 non-null   str    
 13  MonthlyIncome            4410 non-null   int64  
 14  NumCompaniesWorked       4391 non-n

In [60]:
general_data.describe()

,Age,DistanceFromHome,Education,EmployeeCount,EmployeeID,JobLevel,MonthlyIncome,NumCompaniesWorked,PercentSalaryHike,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,YearsAtCompany,YearsSinceLastPromotion,YearsWithCurrManager
count,4410.000000,4410.000000,4410.000000,4410.0,4410.000000,4410.000000,4410.000000,4391.000000,4410.000000,4410.0,4410.000000,4401.000000,4410.000000,4410.000000,4410.000000,4410.000000
mean,36.923810,9.192517,2.912925,1.0,2205.500000,2.063946,65029.312925,2.694830,15.209524,8.0,0.793878,11.279936,2.799320,7.008163,2.187755,4.123129
std,9.133301,8.105026,1.023933,0.0,1273.201673,1.106689,47068.888559,2.498887,3.659108,0.0,0.851883,7.782222,1.288978,6.125135,3.221699,3.567327
min,18.000000,1.000000,1.000000,1.0,1.000000,1.000000,10090.000000,0.000000,11.000000,8.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,30.000000,2.000000,2.000000,1.0,1103.250000,1.000000,29110.000000,1.000000,12.000000,8.0,0.000000,6.000000,2.000000,3.000000,0.000000,2.000000
50%,36.000000,7.000000,3.000000,1.0,2205.500000,2.000000,49190.000000,2.000000,14.000000,8.0,1.000000,10.000000,3.000000,5.000000,1.000000,3.000000
75%,43.000000,14.000000,4.000000,1.0,3307.750000,3.000000,83800.000000,4.000000,18.000000,8.0,1.000000,15.000000,3.000000,9.000000,3.000000,7.000000
max,60.000000,29.000000,5.000000,1.0,4410.000000,5.000000,199990.000000,9.000000,25.000000,8.0,3.000000,40.000000,6.000000,40.000000,15.000000,17.000000


\employee survey data

In [61]:
employee_survey.head()

,EmployeeID,EnvironmentSatisfaction,JobSatisfaction,WorkLifeBalance
0,1,3.0,4.0,2.0
1,2,3.0,2.0,4.0
2,3,2.0,2.0,1.0
3,4,4.0,4.0,3.0
4,5,4.0,1.0,3.0


In [62]:
employee_survey.info()

<class 'pandas.DataFrame'>
RangeIndex: 4410 entries, 0 to 4409
Data columns (total 4 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   EmployeeID               4410 non-null   int64  
 1   EnvironmentSatisfaction  4385 non-null   float64
 2   JobSatisfaction          4390 non-null   float64
 3   WorkLifeBalance          4372 non-null   float64
dtypes: float64(3), int64(1)
memory usage: 137.9 KB


In [63]:
employee_survey.describe()

,EmployeeID,EnvironmentSatisfaction,JobSatisfaction,WorkLifeBalance
count,4410.000000,4385.000000,4390.000000,4372.000000
mean,2205.500000,2.723603,2.728246,2.761436
std,1273.201673,1.092756,1.101253,0.706245
min,1.000000,1.000000,1.000000,1.000000
25%,1103.250000,2.000000,2.000000,2.000000
50%,2205.500000,3.000000,3.000000,3.000000
75%,3307.750000,4.000000,4.000000,3.000000
max,4410.000000,4.000000,4.000000,4.000000


\manager survey data

In [64]:
manager_survey.head()

,EmployeeID,JobInvolvement,PerformanceRating
0,1,3,3
1,2,2,4
2,3,3,3
3,4,2,3
4,5,3,3


In [65]:
manager_survey.info()

<class 'pandas.DataFrame'>
RangeIndex: 4410 entries, 0 to 4409
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   EmployeeID         4410 non-null   int64
 1   JobInvolvement     4410 non-null   int64
 2   PerformanceRating  4410 non-null   int64
dtypes: int64(3)
memory usage: 103.5 KB


In [66]:
manager_survey.describe()

,EmployeeID,JobInvolvement,PerformanceRating
count,4410.000000,4410.000000,4410.000000
mean,2205.500000,2.729932,3.153741
std,1273.201673,0.711400,0.360742
min,1.000000,1.000000,3.000000
25%,1103.250000,2.000000,3.000000
50%,2205.500000,3.000000,3.000000
75%,3307.750000,3.000000,3.000000
max,4410.000000,4.000000,4.000000


\in time data

In [67]:
in_time.iloc[:,:5].head()

,Unnamed: 0,2015-01-01 00:00:00,2015-01-02 00:00:00,2015-01-05 00:00:00,2015-01-06 00:00:00
0,1,NaN,2015-01-02 09:43:45,2015-01-05 10:08:48,2015-01-06 09:54:26
1,2,NaN,2015-01-02 10:15:44,2015-01-05 10:21:05,NaT
2,3,NaN,2015-01-02 10:17:41,2015-01-05 09:50:50,2015-01-06 10:14:13
3,4,NaN,2015-01-02 10:05:06,2015-01-05 09:56:32,2015-01-06 10:11:07
4,5,NaN,2015-01-02 10:28:17,2015-01-05 09:49:58,2015-01-06 09:45:28


\out time data

In [68]:
out_time.iloc[:,:5].head()

,Unnamed: 0,2015-01-01 00:00:00,2015-01-02 00:00:00,2015-01-05 00:00:00,2015-01-06 00:00:00
0,1,NaN,2015-01-02 16:56:15,2015-01-05 17:20:11,2015-01-06 17:19:05
1,2,NaN,2015-01-02 18:22:17,2015-01-05 17:48:22,NaT
2,3,NaN,2015-01-02 16:59:14,2015-01-05 17:06:46,2015-01-06 16:38:32
3,4,NaN,2015-01-02 17:25:24,2015-01-05 17:14:03,2015-01-06 17:07:42
4,5,NaN,2015-01-02 18:31:37,2015-01-05 17:49:15,2015-01-06 17:26:25


\target distribution

In [69]:
general_data["Attrition"].value_counts(normalize=True) *100

Attrition
No     83.877551
Yes    16.122449
Name: proportion, dtype: float64

# relationship between dataset

-- The general_data dataset serves as the primary employee master table, while employee_survey_data and manager_survey_data are integrated using a one-to-one (1:1) relationship on the EmployeeID column.

-- The in_time and out_time datasets contain an unnamed first column representing the EmployeeID, which should be renamed before performing any merge operations.These datasets are joined using a one-to-one (1:1) relationship on both EmployeeID and the corresponding attendance date columns, ensuring accurate pairing of daily check-in and check-out records.